In [0]:
%spark.pyspark 

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
%spark.pyspark

users = spark.createDataFrame(
    [ 
        ("u1", "Berlin"),
        ("u2", "Berlin"),
        ("u3", "Munich"),
        ("u4", "Hamburg"), 
    ],
    ["user_id", "city"] 
)

users.show() 



In [ ]:
%spark.pyspark

orders = spark.createDataFrame(
    [ 
        ("o1", "u1", "p1", 2, 10.0),
        ("o2", "u1", "p2", 1, 30.0),
        ("o3", "u2", "p1", 1, 10.0),
        ("o4", "u2", "p3", 5, 7.0),
        ("o5", "u3", "p2", 3, 30.0),
        ("o6", "u3", "p3", 1, 7.0),
        ("o7", "u4", "p1", 10, 10.0),
    ],
    ["order_id", "user_id", "product_id", "qty", "price"] )

orders.show()


In [ ]:
%spark.pyspark

products = spark.createDataFrame(
    [
        ("p1", "Ring VOLA"),
        ("p2", "Ring POROG"),
        ("p3", "Ring TISHINA"), 
    ],
    ["product_id", "product_name"]
)

products.show()

In [ ]:
%spark.pyspark

# 1. посчитать revenue = qty * price

orders = orders.withColumn("revenue", F.col("qty") * F.col("price"))
orders.show()

In [ ]:
%spark.pyspark

# 2. Объединить orders с users и products

df = (
    orders
        .join(products, "product_id", how="inner")
        .join(users, "user_id", how="inner")
)

df.show()

In [ ]:
%spark.pyspark

# 3. Посчитать метрики по (city, product_id, product_name):
# - orders_cnt (кол-во заказов)
# - qty_sum (сумма qty)
# - revenue_sum (сумма revenue)

metrics = (
    df
        .groupBy("city", "product_id", "product_name")
        .agg(
            F.countDistinct("order_id").alias("orders_cnt"),
            F.sum("qty").alias("qty_sum"),
            F.sum("revenue").alias("revenue_sum")
        )
)

metrics.show()

In [ ]:
%spark.pyspark

# 4. Для каждого города выбрать Top-2 товара по revenue_sum используя Window

w = (
    Window
        .partitionBy("city")
        .orderBy(F.col("revenue_sum").desc())
)

top = (
    metrics
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") <= 2)
        .drop("rn")
)

top.show()

In [ ]:
%spark.pyspark

# 5.1 Записать результат в HDFS по пути /tmp/sandbox_zeppelin/mart_city_top_products/ (parquet, overwrite)
# 6.1  Прочитать обратно и показать топ-результат

(
    top
        .write 
        .mode("overwrite") 
        .parquet("/tmp/sandbox_zeppelin/mart_city_top_products/")
)

spark.read.parquet("/tmp/sandbox_zeppelin/mart_city_top_products/").show()

In [ ]:
%spark.pyspark

# 5.2 Записать результат в s3
# 6.2 Прочитать обратно и показать топ-результат

(
    top
        .write 
        .mode("overwrite") 
        .parquet("s3a://sandbox-zeppelin/mart_city_top_products/") 
)

spark.read.parquet("s3a://sandbox-zeppelin/mart_city_top_products/").show()
